# Modelling: predicting the age of an abalone

This notebook trains a first model that predicts the number of rings of an abalone (its age is roughly the number of rings + 1.5 years) from its physical measurements.

- **Model**: a linear regression, as a simple baseline.
- **Experiment tracking**: each run is logged to MLflow (parameters, test metrics and the trained model), so that runs can be compared in the MLflow UI. MLflow stores its data in `notebooks/mlflow.db` and `notebooks/mlruns/`, which are git-ignored.

To browse the runs, start the MLflow UI from the `notebooks` folder, then open http://127.0.0.1:5001 (port 5000 is often taken by AirPlay on macOS):

```bash
cd notebooks
uv run mlflow ui --port 5001
```

## Imports

In [1]:
import mlflow
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

## Configuration

In [2]:
DATA_PATH = "../data/abalone.csv"
TARGET = "rings"
CATEGORICAL_FEATURE = "sex"
MAX_HEIGHT = 0.5  # taller abalones are data errors (see the EDA)
TEST_SIZE = 0.2
RANDOM_STATE = 42

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"
MLFLOW_EXPERIMENT_NAME = "abalone-age-prediction"

## Data

The dataset is the `abalone.csv` file from Kaggle, saved in the git-ignored `data/` folder. Column names are converted to snake_case (e.g. `Whole weight` becomes `whole_weight`) to make them easier to use in code.

In [3]:
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.lower().str.replace(" ", "_")

print(f"{df.shape[0]} abalones, {df.shape[1]} columns")
df.head()

4177 abalones, 9 columns


,sex,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


## Cleaning

The EDA found 4 rows with an impossible or extreme height: 2 abalones with a height of 0, and 2 taller than 0.5, when all the others are at most 0.25. These are data errors, so they are removed before training.

In [4]:
def remove_height_outliers(df: pd.DataFrame) -> pd.DataFrame:
    """Drop the rows with an impossible (zero) or extreme height."""
    return df[(df["height"] > 0) & (df["height"] <= MAX_HEIGHT)]


clean_df = remove_height_outliers(df)
print(f"Removed {len(df) - len(clean_df)} rows, {len(clean_df)} abalones left")

Removed 4 rows, 4173 abalones left


## Preprocessing

The data is split into a training set and a test set. The only non-numeric column is `sex` (`M`, `F`, or `I` for infant), which is one-hot encoded. The encoder is fitted on the training set only, then reused as-is on the test set, as it will be when making predictions on new data. `drop="first"` removes one redundant column, which a linear regression needs to avoid perfectly correlated features.

In [5]:
def split_data(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Split the dataset into train and test features and targets."""
    features = df.drop(columns=TARGET)
    target = df[TARGET]
    return train_test_split(
        features, target, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )


def encode_categorical(
    features: pd.DataFrame, encoder: OneHotEncoder | None = None
) -> tuple[pd.DataFrame, OneHotEncoder]:
    """One-hot encode the categorical feature, fitting an encoder if none is given."""
    if encoder is None:
        encoder = OneHotEncoder(drop="first", sparse_output=False)
        encoder.set_output(transform="pandas")
        encoder.fit(features[[CATEGORICAL_FEATURE]])
    encoded = encoder.transform(features[[CATEGORICAL_FEATURE]])
    features = pd.concat([features.drop(columns=CATEGORICAL_FEATURE), encoded], axis=1)
    return features, encoder


X_train, X_test, y_train, y_test = split_data(clean_df)
X_train, encoder = encode_categorical(X_train)
X_test, _ = encode_categorical(X_test, encoder)

print(f"Training set: {len(X_train)} rows, test set: {len(X_test)} rows")
X_train.head()

Training set: 3338 rows, test set: 835 rows


,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,sex_I,sex_M
554,0.485,0.390,0.125,0.5910,0.2870,0.1410,0.1200,1.0,0.0
3074,0.675,0.520,0.145,1.3645,0.5570,0.3405,0.3850,0.0,1.0
2405,0.655,0.515,0.145,1.2500,0.5265,0.2830,0.3150,1.0,0.0
1956,0.645,0.490,0.175,1.3200,0.6525,0.2375,0.3385,0.0,1.0
1947,0.635,0.500,0.165,1.2730,0.6535,0.2130,0.3650,0.0,1.0


## Training and tracking with MLflow

The model is trained on the training set and evaluated on the test set with three metrics:
- **RMSE** (root mean squared error) and **MAE** (mean absolute error), both in number of rings: the lower, the better.
- **R²**: the share of the variation in the number of rings that the model explains (1 is perfect).

In [6]:
def evaluate_model(
    model: LinearRegression, features: pd.DataFrame, target: pd.Series
) -> dict[str, float]:
    """Compute the model's regression metrics on the given data."""
    predictions = model.predict(features)
    return {
        "rmse": root_mean_squared_error(target, predictions),
        "mae": mean_absolute_error(target, predictions),
        "r2": r2_score(target, predictions),
    }


mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="linear_regression"):
    model = LinearRegression()
    model.fit(X_train, y_train)
    test_metrics = evaluate_model(model, X_test, y_test)

    mlflow.log_params(
        {
            "model": "LinearRegression",
            "max_height": MAX_HEIGHT,
            "test_size": TEST_SIZE,
            "random_state": RANDOM_STATE,
        }
    )
    mlflow.log_metrics({f"test_{name}": value for name, value in test_metrics.items()})
    mlflow.sklearn.log_model(model, name="model", input_example=X_train.head())

pd.Series(test_metrics, name="test set").round(3)

2026/10/22 12:17:08 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


rmse    2.229
mae     1.584
r2      0.585
Name: test set, dtype: float64